# Deep learning models

Run the setup and config cells once, then edit and re-run the experiment cell. Every run is logged to `training_runs/results.csv`.

In [ ]:
import sys
sys.path.append("../src")

%load_ext autoreload
%autoreload 2

from dataclasses import dataclass, field
import pandas as pd
from dl_runner import run_dl

In [ ]:
@dataclass
class RunConfig:
    """Settings every run needs."""
    experiment_name: str = "baseline"
    notes: str = ""                  # why you tried this run
    data_dir: str = "data"
    output_dir: str = "training_runs"
    seed: int = 42
    evaluate_test: bool = False      # keep False while iterating


@dataclass
class DLConfig(RunConfig):
    """Settings for the PyTorch sequence models."""
    model_name: str = "gru"          # lstm, gru, cnn_lstm
    model_params: dict = field(default_factory=dict)   # hidden_size, num_layers, dropout (cnn_lstm: cnn_filters, kernel_size)
    sequence_length: int = 72        # rows of history per sample: 24 = 4 h, 72 = 12 h, 144 = 24 h
    epochs: int = 50
    batch_size: int = 128
    learning_rate: float = 1e-3
    weight_decay: float = 1e-4
    loss: str = "huber"              # mse, huber, mae
    huber_delta: float = 1.0
    patience: int = 8
    monitor: str = "rmse"            # rmse, mae, mape, loss
    grad_clip: float = 1.0           # 0 = off

## Experiment

In [ ]:
config = DLConfig(
    experiment_name="gru_base",
    model_name="gru",
    model_params={"hidden_size": 64, "num_layers": 1, "dropout": 0.2},
    sequence_length=72,
    epochs=50,
    learning_rate=1e-3,
    loss="huber",
    notes="first GRU on the new pipeline",
)
run_dl(config)

## Results so far

In [ ]:
results = pd.read_csv("training_runs/results.csv")
results.reindex(columns=["when", "experiment", "model", "sequence_length", "val_mae", "val_rmse", "val_r2", "best_epoch", "notes"])